# FinSentinel — EDA & Pipeline Walkthrough

This notebook walks through every stage of the FinSentinel pipeline with
visualisations and commentary. Use it to:
- Understand how each module works
- Explore scraped data quality
- Inspect FinBERT sentiment distributions
- Audit signal generation logic
- Review backtest equity curves interactively

**Pipeline:** `Scraper → Preprocessor → FinBERT → Signals → Backtest`

## 0 — Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')   # add project root to path

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.style.use('dark_background')
plt.rcParams.update({'figure.figsize': (14, 5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.family': 'monospace'})

from src.utils import load_config
cfg = load_config('../config.yaml')
print('Tickers:', cfg.tickers)
print('Date range:', cfg.date_range.start, '→', cfg.date_range.end)

## 1 — Scraper: Fetching News & Filings

The scraper pulls from two sources:
1. **Yahoo Finance RSS** — real-time headlines, no API key required
2. **SEC EDGAR submissions API** — 8-K filings for earnings events

Rate limiting (1 req/sec) and retry logic (3 attempts, exponential backoff)
are handled transparently by `RateLimiter` and the `@retry` decorator.

In [ ]:
from src.scraper import scrape_all

# Scrape all configured tickers (this takes ~30s due to rate limiting)
raw_df = scrape_all(cfg=cfg, save=True)
print(f'Total articles scraped: {len(raw_df)}')
raw_df.head(10)

In [ ]:
# Article count by ticker and source
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

raw_df.groupby('ticker').size().plot(kind='bar', ax=axes[0],
    color='#58a6ff', title='Articles per Ticker')
axes[0].set_xlabel('')

raw_df.groupby('source').size().plot(kind='bar', ax=axes[1],
    color='#3fb950', title='Articles by Source')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

# Timeline of articles
raw_df['day'] = pd.to_datetime(raw_df['date']).dt.floor('D')
timeline = raw_df.groupby(['day', 'ticker']).size().unstack(fill_value=0)
timeline.plot(figsize=(14, 4), title='Daily Article Volume by Ticker')
plt.ylabel('Article Count')
plt.tight_layout()
plt.show()

## 2 — Preprocessor: Cleaning Text

Steps applied in order:
1. Strip HTML tags & decode entities
2. Remove URLs
3. Remove rare special characters
4. Deduplicate by `(ticker, headline)`
5. Keyword filter (removes ads, sponsored content)
6. Build combined `text` field: `headline + ". " + summary`
7. Drop articles shorter than `min_article_length` tokens

In [ ]:
from src.preprocessor import preprocess

clean_df = preprocess(raw_df, cfg=cfg)
print(f'Before: {len(raw_df)} → After: {len(clean_df)} articles')
print(f'Drop rate: {(1 - len(clean_df)/len(raw_df))*100:.1f}%')
clean_df[['ticker', 'date', 'text']].head(5)

In [ ]:
# Token length distribution
clean_df['token_count'] = clean_df['text'].str.split().str.len()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(clean_df['token_count'], bins=50, color='#58a6ff', alpha=0.8)
ax.axvline(512, color='#f85149', linestyle='--', label='FinBERT max (512)')
ax.set_xlabel('Token Count')
ax.set_ylabel('Articles')
ax.set_title('Article Length Distribution (after preprocessing)')
ax.legend()
plt.tight_layout()
plt.show()

print(clean_df['token_count'].describe().round(1))

## 3 — FinBERT Sentiment Analysis

**Model:** `ProsusAI/finbert` — fine-tuned on financial text (10K/8K filings,
earnings call transcripts, financial news). Outputs three classes:
- `positive` — bullish tone
- `negative` — bearish tone  
- `neutral`  — factual / no clear direction

**Daily aggregation:**
- `daily_score = weighted_avg(pos - neg)` scaled to [0,1], weight = 2× for <24h old articles
- `sentiment_momentum = 3-day rolling change in daily_score`

In [ ]:
from src.sentiment import analyze

# This will download FinBERT on first run (~500MB)
article_df, daily_df = analyze(clean_df, cfg=cfg)
print('Article-level scores:')
article_df[['ticker', 'date', 'headline', 'pos', 'neg', 'neu', 'sentiment_label']].head(8)

In [ ]:
# Sentiment label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

label_counts = article_df.groupby(['ticker', 'sentiment_label']).size().unstack(fill_value=0)
label_counts.plot(kind='bar', ax=axes[0], stacked=True,
    color=['#3fb950', '#f85149', '#8b949e'],
    title='Sentiment Labels by Ticker')
axes[0].set_xlabel('')
axes[0].legend(['Positive', 'Negative', 'Neutral'])

# Score distributions per ticker
for ticker in cfg.tickers:
    subset = article_df[article_df['ticker'] == ticker]['pos'] - article_df[article_df['ticker'] == ticker]['neg']
    axes[1].hist(subset, bins=30, alpha=0.6, label=ticker)
axes[1].axvline(0, color='white', linestyle='--', alpha=0.5)
axes[1].set_xlabel('pos − neg score')
axes[1].set_title('Net Sentiment Distribution per Ticker')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Daily sentiment score timeline
fig, axes = plt.subplots(len(cfg.tickers), 1,
                         figsize=(14, 3 * len(cfg.tickers)), sharex=True)

colors = ['#58a6ff', '#3fb950', '#f85149', '#d2a8ff', '#ffa657']
for ax, ticker, color in zip(axes, cfg.tickers, colors):
    d = daily_df[daily_df['ticker'] == ticker].sort_values('date')
    ax.fill_between(d['date'], d['daily_score'], alpha=0.2, color=color)
    ax.plot(d['date'], d['daily_score'], color=color, linewidth=1.5, label=ticker)
    ax.axhline(0.6, color='#3fb950', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.axhline(0.4, color='#f85149', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.legend(loc='upper right')
    ax.set_title(f'{ticker} Daily Sentiment Score')

plt.suptitle('Daily Sentiment Scores — BUY (green) / SELL (red) thresholds', y=1.01)
plt.tight_layout()
plt.show()

## 4 — Signal Generation

Signal rules (all vectorized — no Python loops):

| Condition | Signal |
|---|---|
| `daily_score > 0.6` AND `momentum > 0.1` | **BUY** |
| `daily_score < 0.4` AND `momentum < -0.1` | **SELL** |
| otherwise | **HOLD** |

**Smoothing:** signal must persist for 2 consecutive days before acting.
**Confidence:** proportional to distance from threshold.

In [ ]:
from src.signals import generate_signals, get_latest_signals

signal_df = generate_signals(daily_df, cfg=cfg)
print('Signal distribution:')
print(signal_df['signal'].value_counts())
print()
print('Latest signals:')
get_latest_signals(signal_df)

In [ ]:
# Signal timeline per ticker
signal_colors = {'BUY': '#3fb950', 'SELL': '#f85149', 'HOLD': '#8b949e'}

fig, axes = plt.subplots(len(cfg.tickers), 1,
                         figsize=(14, 3 * len(cfg.tickers)), sharex=True)

for ax, ticker in zip(axes, cfg.tickers):
    d = signal_df[signal_df['ticker'] == ticker].sort_values('date')
    ax.plot(d['date'], d['raw_score'], color='#58a6ff', linewidth=1.2, alpha=0.7, label='Score')
    for sig, color in signal_colors.items():
        mask = d['signal'] == sig
        if mask.any():
            ax.scatter(d.loc[mask, 'date'], d.loc[mask, 'raw_score'],
                       color=color, s=60, zorder=5, label=sig)
    ax.axhline(0.6, color='#3fb950', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.axhline(0.4, color='#f85149', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.set_title(f'{ticker} — Signals on Sentiment Score')
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 5 — Backtesting

**Strategy parameters:**
- Starting capital: **$100,000**
- Position sizing: **Half-Kelly Criterion** (clamped 5%–25%)
- Entry: next day's OPEN after BUY signal
- Exit: SELL signal OR 10-day max hold
- Transaction cost: **0.1%** per side
- Benchmark: equal-weight Buy-and-Hold

In [ ]:
from src.backtest import run_backtest
import pprint

equity_curve, metrics, trade_log = run_backtest(signal_df, cfg=cfg)

print('=== Performance Metrics ===')
pprint.pprint(metrics)

In [ ]:
# Equity curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.plot(equity_curve['date'], equity_curve['portfolio_value'],
        color='#58a6ff', linewidth=2, label='FinSentinel Strategy')
ax.plot(equity_curve['date'], equity_curve['benchmark_value'],
        color='#8b949e', linewidth=1.5, linestyle='--', label='Buy & Hold Benchmark')
ax.set_ylabel('Portfolio Value ($)')
ax.set_title('FinSentinel — Equity Curve vs Benchmark')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()

# Drawdown
ax2 = axes[1]
ax2.fill_between(equity_curve['date'], equity_curve['drawdown'] * 100,
                 0, alpha=0.5, color='#f85149')
ax2.plot(equity_curve['date'], equity_curve['drawdown'] * 100,
         color='#f85149', linewidth=1)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.show()

In [ ]:
# Trade log analysis
print(f'Total trades: {len(trade_log)}')
print()

if not trade_log.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # P&L distribution
    colors_pnl = ['#3fb950' if x > 0 else '#f85149' for x in trade_log['pnl_pct']]
    axes[0].bar(range(len(trade_log)), trade_log['pnl_pct'] * 100, color=colors_pnl)
    axes[0].set_xlabel('Trade #')
    axes[0].set_ylabel('P&L %')
    axes[0].set_title('Trade P&L (%)')
    axes[0].axhline(0, color='white', linewidth=0.8)

    # Hold duration
    hold_days = (pd.to_datetime(trade_log['exit_date']) -
                 pd.to_datetime(trade_log['entry_date'])).dt.days
    axes[1].hist(hold_days, bins=10, color='#d2a8ff', alpha=0.8)
    axes[1].set_xlabel('Hold Duration (days)')
    axes[1].set_title('Hold Duration Distribution')

    # Confidence vs P&L
    axes[2].scatter(trade_log['signal_confidence'],
                    trade_log['pnl_pct'] * 100,
                    c=colors_pnl, alpha=0.7, s=60)
    axes[2].set_xlabel('Signal Confidence')
    axes[2].set_ylabel('P&L %')
    axes[2].set_title('Confidence vs P&L')
    axes[2].axhline(0, color='white', linewidth=0.8)

    plt.tight_layout()
    plt.show()

    print(trade_log[['ticker','entry_date','exit_date','pnl_pct','exit_reason']].to_string())

## 6 — Summary

| Stage | Output |
|---|---|
| `scraper.py` | Raw news + 8-K filings DataFrame |
| `preprocessor.py` | Clean, deduplicated text with UTC dates |
| `sentiment.py` | Per-article pos/neg/neu + daily_score + momentum |
| `signals.py` | BUY/SELL/HOLD + confidence (vectorized, smoothed) |
| `backtest.py` | Equity curve, 12 metrics, trade log |
| `dashboard.py` | Interactive 3-page Streamlit UI |

### Next Steps
- Add options sentiment (implied volatility skew signals)
- Integrate alternative data: earnings call transcripts, social media
- Live paper-trading mode via Alpaca/IBKR API
- Walk-forward cross-validation for out-of-sample testing

> ⚠️ **Disclaimer:** For educational purposes only. Not financial advice.